# Lesson 1: Your Study Area

**Planning question: What is my study area?**

Every planning project starts by defining the area you are working in. In this notebook you will:

1. Download the official boundary of Bengaluru
2. Measure its area
3. Load the BBMP ward boundaries
4. Make your first maps

Part of [RS & GIS for Spatial Planners](https://soniadas123.github.io/rs-gis-spatial-planner/).
Run each cell in order with `Shift + Enter`.

## Step 0: Install the tools

Colab already has most of what we need. This cell adds a few extra packages and takes about a minute.

In [ ]:
!pip install -q osmnx

## Step 1: Load the packages

- **geopandas** works with map data stored as tables
- **osmnx** downloads data from OpenStreetMap
- **matplotlib** draws static maps and charts

In [ ]:
import geopandas as gpd
import osmnx as ox
import matplotlib.pyplot as plt

## Step 2: Settings

Change these two lines to work on a different city.
`UTM_CRS` is a coordinate system that measures in metres. EPSG:32643 (UTM zone 43N) suits Bengaluru.

In [ ]:
CITY = "Bengaluru, Karnataka, India"
UTM_CRS = "EPSG:32643"

## Step 3: Download the city boundary

`geocode_to_gdf` looks up the name in OpenStreetMap and returns its boundary as a **GeoDataFrame**:
a table with a `geometry` column that holds the shape.

In [ ]:
city = ox.geocode_to_gdf(CITY)
city[["display_name", "geometry"]]

In [ ]:
city.plot(figsize=(6, 6), color="lightblue", edgecolor="black")
plt.title("Bengaluru city boundary")
plt.show()

## Step 4: Measure the area

Latitude and longitude are measured in degrees, so they cannot give an area in square kilometres.
We first **reproject** the boundary to UTM, which uses metres, and then calculate the area.

In [ ]:
city_utm = city.to_crs(UTM_CRS)
area_km2 = city_utm.area.iloc[0] / 1_000_000
print(f"Area of {CITY}: {area_km2:,.0f} square km")

## Step 5: Load the ward boundaries

Wards are the smallest administrative unit used for planning and elections in Bengaluru.
DataMeet, a community of data enthusiasts, shares the BBMP ward boundaries as open data.
We can read the file straight from the internet.

In [ ]:
WARDS_URL = "https://raw.githubusercontent.com/datameet/Municipal_Spatial_Data/master/Bangalore/BBMP.geojson"
wards = gpd.read_file(WARDS_URL)
print("Number of wards:", len(wards))
wards.head()

Each row is one ward. `KGISWardName` holds the ward name.
Next we add a column with each ward's area.

In [ ]:
wards_utm = wards.to_crs(UTM_CRS)
wards["area_km2"] = wards_utm.area / 1_000_000

wards[["KGISWardName", "area_km2"]].sort_values("area_km2", ascending=False).head(10)

## Step 6: Map ward size

A **choropleth map** colours each area by a value. Here, darker colours mean larger wards.
Large wards are usually on the edge of the city, where buildings are spread out.
In Lesson 10 you will learn to turn maps like this into interactive web maps that you can zoom and click.

In [ ]:
ax = wards.plot(column="area_km2", cmap="YlGnBu", legend=True, figsize=(8, 8),
                edgecolor="white", linewidth=0.3,
                legend_kwds={"label": "Ward area (square km)"})
ax.set_title("BBMP wards by area")
ax.set_axis_off()
plt.show()

## Step 7: Save your data

GeoPackage (`.gpkg`) is a single-file format that QGIS and ArcGIS can open.
After running the cell, find the file in the **Files** panel on the left of Colab, then right-click it and choose **Download**.

In [ ]:
city.to_file("bengaluru_study_area.gpkg", layer="city_boundary", driver="GPKG")
wards.to_file("bengaluru_study_area.gpkg", layer="wards", driver="GPKG")
print("Saved bengaluru_study_area.gpkg")

## What this means for planning

- A clear study area boundary is the base of every planning analysis.
- Wards let you compare neighbourhoods and report results in units that decision makers know.
- Always use a projected coordinate system (in metres) when you measure areas or distances.

## Try it yourself

1. Change `CITY` to `"Mysuru, Karnataka, India"` and run Steps 3 and 4 again. How does its area compare?
2. Find the smallest ward. Hint: change `ascending=False` to `ascending=True`.
3. In Step 6, change `cmap="YlGnBu"` to `"OrRd"` or `"viridis"`. Which colours make the map easiest to read?

---
*Enjoying the course? Support it with a [star on GitHub](https://github.com/soniadas123/rs-gis-spatial-planner).*